In [1]:
import os
import sys
import numpy as np
import moeabench as mb
import pandas as pd
sys.path.append(os.path.abspath("."))
from src.meamt_cantos import CCSGA
from src.meamt_allmasks_wrapper import MULTICORNER
from src.maoeacs import MAOEACS
from src.maf2 import MaF2
from src.tabelas_cantos_wrapper import TabelasCantos

In [ ]:
nobj = 10
pop = 275
ngen = 1000
repeats = 1 

# DTLZ1 ao DTLZ9 formam a base clássica da suíte
problemas = {f"DPF{i}": getattr(mb.mops, f"DPF{i}")(M=nobj, seed=42) for i in range(3, 3)}
problemas.update({f"DTLZ{i}": getattr(mb.mops, f"DTLZ{i}")(M=nobj, seed=42) for i in range(1, 2)})
#problemas.update({"MaF2": MaF2(M=nobj)})
experimentos_salvos = {}

print(f"Iniciando bateria de testes ({nobj} Objetivos, Pop: {pop}, Gen: {ngen})...")

for nome_prob, prob in problemas.items():
    print(f"Processando {nome_prob}...")

    competidores = {
        #"MULTICORNER": MULTICORNER(population=pop, generations=ngen),
        #"CCSGA": CCSGA(population=pop, generations=ngen),
        #"MEAMT": MEAMT(population=pop, generations=ngen, tabelas=[(1,)], incluir_completa=False),
        #"NSGA-II": mb.moeas.NSGA2(population=pop, generations=ngen),
        #"MAOEACS": MAOEACS(population=pop, generations=ngen),
        #"MOEAD": mb.moeas.MOEAD(population=pop, generations=ngen),
        #"NSGA-III": mb.moeas.NSGA3(population=pop, generations=ngen),
        #"RVEA": mb.moeas.RVEA(population=pop, generations=ngen),
        "CANTOS": TabelasCantos(population=pop, generations=ngen) 
    }

    lista_exps = []
    for nome_algo, instancia_algo in competidores.items():
        exp = mb.experiment()
        exp.name = nome_algo
        exp.mop = prob
        exp.moea = instancia_algo
        
        # Execução silenciosa para não poluir o terminal
        exp.run(repeat=repeats, silent=False)
        lista_exps.append(exp)

    experimentos_salvos[nome_prob] = lista_exps

print("Todas as execuções finalizadas e salvas em memória!")




Iniciando bateria de testes (10 Objetivos, Pop: 275, Gen: 700)...
Processando DTLZ1...


### Running **CANTOS**

  Run 1/1:   0%|          | 0/700 [00:00<?, ?gen/s]

Todas as execuções finalizadas e salvas em memória!


In [3]:
linhas = []

# Cria uma população sintética contendo apenas os pontos delimitadores

for nome_prob, lista_exps in experimentos_salvos.items():
    for exp in lista_exps:
        hv = float('nan')
        igd_plus = float('nan')
        
        try:
            # Garanta que o objeto extraído seja uma matriz 2D das soluções finais
            # Exemplo: exp.result ou a população final
            hv_matrix = mb.metrics.hypervolume(exp, ref=lista_exps)
            igd_matrix = mb.metrics.igdplus(exp)

            hv = float(np.array(hv_matrix).flatten()[-1])
            igd_plus = float(np.array(igd_matrix).flatten()[-1])

        except Exception as e:
            print(f"Aviso: Não foi possível extrair métricas para {exp.name} no {nome_prob}. Erro: {e}")

        linhas.append({
            "Problema": nome_prob,
            "Algoritmo": exp.name,
            "HV": hv,
            "IGD+": igd_plus
        })

df_resultados = pd.DataFrame(linhas)
display(df_resultados)

Computing Hypervolume (CANTOS):   0%

Computing IGD+ (CANTOS):   0%

/tmp/ipykernel_55875/2269799723.py:16: DeprecationWarning: __array__ implementation doesn't accept a copy keyword, so passing copy=False failed. __array__ must implement 'dtype' and 'copy' keyword arguments. To learn more, see the migration guide https://numpy.org/devdocs/numpy_2_0_migration_guide.html#adapting-to-changes-in-the-copy-keyword
  hv = float(np.array(hv_matrix).flatten()[-1])
/tmp/ipykernel_55875/2269799723.py:17: DeprecationWarning: __array__ implementation doesn't accept a copy keyword, so passing copy=False failed. __array__ must implement 'dtype' and 'copy' keyword arguments. To learn more, see the migration guide https://numpy.org/devdocs/numpy_2_0_migration_guide.html#adapting-to-changes-in-the-copy-keyword
  igd_plus = float(np.array(igd_matrix).flatten()[-1])


,Problema,Algoritmo,HV,IGD+
0,DTLZ1,CANTOS,2.593038,0.264247


In [5]:
# Escolha qual problema da suíte DTLZ você quer visualizar agora
alvo = "DTLZ1" # Altere para o problema desejado, por exemplo, "DPF1", "DPF2", etc.

# Extrai a lista com os 5 experimentos (MEAMT, NSGA-II, NSGA-III, MOEA/D, SPEA2)
exps_alvo = experimentos_salvos[alvo]

# Extrai os nomes automaticamente para usar na legenda
nomes_algoritmos = [exp.name for exp in exps_alvo]

# Renderiza a topologia iterativa nativa do moeabench
mb.view.topology(
    *exps_alvo, 
    labels=nomes_algoritmos, 
    title=f'Comparação Topológica do Pareto - {alvo} ({nobj} Objetivos projetados em 3D)'
)

